# ParticleTracer event displays

Reads the ROOT files written by `Offline/STMMC/src/ParticleTracer_module.cc`
(fcl: `Offline/STMMC/fcl/ParticleTracer.fcl`) and draws one top/side view event display
per art event.

One TTree entry = one `SimParticle`: either a particle that reached the seeded
StepPointMCs and passed the PDG filter (`matched == True`) or one of its ancestors up
to the primary (`matched == False`).

* **solid line** - the entry has a stored `MCTrajectory` (`hasTrajectory == True`); the
  line runs through all of its points.
* **dashed line** - no `MCTrajectory` was stored (the particle failed the G4 trajectory
  cuts), so the entry only holds the `SimParticle` start and end positions and the line
  is a straight segment between them.

Colour is by PDG ID (`pdgid.pdgid_color_dict`); seeded particles are drawn slightly
thicker than their ancestors.

In [ ]:
from __future__ import print_function
import sys, os
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import ROOT
from ROOT import gROOT, gStyle, gDirectory, gPad

import filepath
import portROOT2pd_particletracer
from pdgid import pdgid_dict
import plot_utils
import constants

geometry = "MDC2025ab"
tags = ["Ele"]
pklname = geometry + "_particletracer.pkl"

# Output of ParticleTracer.fcl. Add entries here (or to filepath.py) as more are produced.
particle_tracer_root_files = {
    "MDC2025ab": {
        "Ele": ["/exp/mu2e/data/users/yongyiwu/MDC2025ab/datasets/MDC2025/rootfiles/BeamToEleVD101Neutron.root"],
    }
}

In [ ]:
df_traj = pd.DataFrame()
for tag in tags:
    fileList_ = particle_tracer_root_files[geometry][tag]
    dft_ = portROOT2pd_particletracer.PortToDF(geometry, tag, fileList_, verbose=True,
                                               treedir="particleTracer", treename="ttree",
                                               weighted=False)
    df_traj = pd.concat([df_traj, dft_], ignore_index=True)
with open(pklname, 'wb') as f:
    pickle.dump(df_traj, f)

In [ ]:
with open(pklname, 'rb') as f:
    df_traj = pickle.load(f)
display(df_traj)

print("entries:              ", len(df_traj))
print("with MCTrajectory:    ", int(df_traj['hasTrajectory'].sum()), "(solid)")
print("start/end only:       ", int((~df_traj['hasTrajectory']).sum()), "(dashed)")
print("matched (seed) parts: ", int(df_traj['matched'].sum()))
print("ancestors:            ", int((~df_traj['matched']).sum()))
print("pdgIds present:       ", np.sort(df_traj['pdgId'].unique()))

df_events = portROOT2pd_particletracer.getEventList(df_traj)
print("art events:           ", len(df_events))
display(df_events)

## Per-event displays

One figure per art event, showing every trajectory entry of that event (seeded particles
plus their full ancestry). Change `nshow`/`stride` to scan more or fewer events.

In [ ]:
nshow = 10   # how many events to draw
stride = 1   # step through the event list

for ii in range(0, min(nshow*stride, len(df_events)), stride):
    print('------------------------------------------------------------------------------')
    ev_ = df_events.iloc[ii]
    dfe_ = portROOT2pd_particletracer.getEvent(df_traj, ev_['tag'], ev_['fileno'],
                                               ev_['run'], ev_['subRun'], ev_['event'])
    display(dfe_[['simId', 'pdgId', 'matched', 'hasTrajectory', 'nPoints',
                  'parentSimId', 'parentPdgId', 'creationCode', 'isPrimary',
                  'startx', 'starty', 'startz', 'starttime', 'startkE',
                  'endx', 'endy', 'endz', 'endtime', 'endkE']])
    title = ("ParticleTracer " + str(ev_['tag']) + " %03i" % ev_['fileno'] +
             "  run %i subRun %i event %i" % (ev_['run'], ev_['subRun'], ev_['event']) +
             "  (%i trajectories)" % len(dfe_))
    fig, ax_top, ax_side = plot_utils.draw_particle_tracer_event(dfe_, title)
    plt.show()

## One display per seeded particle

An event can contain several seeded particles. This draws one figure per
`matched == True` entry, showing that particle together with only its own ancestor chain
(`ancestorSimIds`), which is easier to read when the event is busy.

In [ ]:
df_matched = df_traj.query("matched == True").reset_index(drop=True)
print(len(df_matched), " seeded particles")

nshow = 10
stride = 1

for ii in range(0, min(nshow*stride, len(df_matched)), stride):
    print('------------------------------------------------------------------------------')
    seed_ = df_matched.iloc[ii]
    dfe_ = portROOT2pd_particletracer.getEvent(df_traj, seed_['tag'], seed_['fileno'],
                                               seed_['run'], seed_['subRun'], seed_['event'])
    # the seed itself plus its ancestor chain (parent-first, primary last)
    chain = [seed_['simId']] + list(seed_['ancestorSimIds'])
    present = set(dfe_['simId'])
    dfc_ = dfe_.set_index('simId').loc[[s for s in chain if s in present]].reset_index()
    display(dfc_[['simId', 'pdgId', 'matched', 'hasTrajectory', 'nPoints',
                  'parentSimId', 'parentPdgId', 'creationCode', 'isPrimary',
                  'startx', 'starty', 'startz', 'starttime', 'startkE',
                  'endx', 'endy', 'endz', 'endtime', 'endkE']])
    try:
        seed_name = pdgid_dict[seed_['pdgId']]
    except KeyError:
        seed_name = str(int(seed_['pdgId']))
    title = ("Back trace of " + str(seed_['tag']) + " %03i " % seed_['fileno'] + seed_name +
             "  run %i subRun %i event %i simId %i" % (seed_['run'], seed_['subRun'],
                                                       seed_['event'], seed_['simId']))
    fig, ax_top, ax_side = plot_utils.draw_particle_tracer_event(dfc_, title)
    plt.show()

## Save the displays to a PDF

In [ ]:
nsave = 20
pdfname = geometry + "_particletracer_displays.pdf"
with PdfPages(pdfname) as pdf:
    for ii in range(min(nsave, len(df_events))):
        ev_ = df_events.iloc[ii]
        dfe_ = portROOT2pd_particletracer.getEvent(df_traj, ev_['tag'], ev_['fileno'],
                                                   ev_['run'], ev_['subRun'], ev_['event'])
        title = ("ParticleTracer " + str(ev_['tag']) + " %03i" % ev_['fileno'] +
                 "  run %i subRun %i event %i" % (ev_['run'], ev_['subRun'], ev_['event']))
        fig, ax_top, ax_side = plot_utils.draw_particle_tracer_event(dfe_, title)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
print("written " + pdfname)